# DSCI 552 - Homework 3

- **Name:** Jesse Fulcher
- **GitHub Username:** jfulch
- **USC ID:** 7451-9585-45

## 1 - Time Series Classification: Featue Creation and Extraction

### 1(a) - AReM Dataset

The AReM dataset contains 88 instances from seven human activities: bending1, bending2, cycling, lying, sitting standing, and walking.

Each instance contains 480 consecutive observations of six time series:

- `avg_rss12`
- `var_rss12`
- `avg_rss13`
- `var_rss13`
- `avg_rss23`
- `var_rss23`

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

data_directory = Path("../data/AReM")

activity_directories = sorted(
    path
    for path in data_directory.iterdir()
    if path.is_dir()
)

dataset_summary = pd.DataFrame({
    "Activity": [path.name for path in activity_directories],
    "Number of Instances": [
        len(list(path.glob("dataset*.csv")))
        for path in activity_directories
    ],
})

dataset_summary.loc[len(dataset_summary)] = [
    "Total",
    dataset_summary["Number of Instances"].sum(),
]

dataset_summary

,Activity,Number of Instances
0,bending1,7
1,bending2,6
2,cycling,15
3,lying,15
4,sitting,15
5,standing,15
6,walking,15
7,Total,88


### 1(b) - Training and Test Data

For bending1 and bending2, datasets 1 and 2 are assigned to the test set. For all other activities, datasets 1, 2 and 3 are assigned to the test set. The remaining datasets are assigned to the training set.

In [10]:
split_records = []

for activity_directory in activity_directories:
    activity = activity_directory.name

    dataset_paths = sorted(
        activity_directory.glob("dataset*.csv"),
        key=lambda path: int(path.stem.replace("dataset", "")),
    )

    test_dataset_count = (
        2 if activity in {"bending1", "bending2"} else 3
    )

    for dataset_path in dataset_paths:
        dataset_number = int(
            dataset_path.stem.replace("dataset", "")
        )

        split = (
            "Test"
            if dataset_number <= test_dataset_count
            else "Train"
        )

        split_records.append({
            "Activity": activity,
            "Dataset": dataset_number,
            "Path": dataset_path,
            "Split": split,
        })

split_manifest = pd.DataFrame(split_records)

train_files = split_manifest.loc[
    split_manifest["Split"] == "Train", "Path"
].tolist()

test_files = split_manifest.loc[
    split_manifest["Split"] == "Test", "Path"
].tolist()

split_summary = pd.crosstab(
    split_manifest["Activity"],
    split_manifest["Split"],
    margins=True,
)

split_summary

Split,Test,Train,All
Activity,,,
bending1,2,5,7
bending2,2,4,6
cycling,3,12,15
lying,3,12,15
sitting,3,12,15
standing,3,12,15
walking,3,12,15
All,19,69,88


### 1(c) - Feature Extraction

#### 1(c)(i) - Common Time-Domain Features

Common time-domain features used in time-series classification include:

- Minimum and maximum
- Mean and median
- Range
- Variance and standard deviation
- First and third quartiles
- Interquartile range
- Root mean square
- Energy
- Mean absolute deviation
- Skewness
- Kurtosis
- Zero-crossing and mean-crossing rates
- Peak count and peak amplitude
- Autocorrelation
- Signal entropy
- Trend or slope

These features summarize characteristics such as central tendency, variability, distribution shape, signal magnitude, temporal dependence, and changes over time. For the remainder of this assignment, the required features are minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### 1(c)(ii) - Extracted Time-Domain Features

For each of the 88 instances, the minimum, maximum, mean, median, standard deviation, first quartile and third quartile were calculated for each of the six time series. This produces \(7 \times 6=42\) features per instance.

In [11]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12 ",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23",
]

time_series_columns = column_names[1:]

def load_instance(dataset_path):
    data =pd.read_csv(
        dataset_path,
        comment="#",
        header=None,
        names=column_names,
        sep=r"\s+|,",
        engine="python",
        usecols=range(7)
    )
    return data.apply(pd.to_numeric, errors="raise")


feature_record = []
instance_record = []

ordered_instances = split_manifest.sort_values(
    ["Activity", "Dataset"]
).reset_index(drop=True)

for instance_number, row in ordered_instances.iterrows():
    instance_data = load_instance(row["Path"])
    instance_features = {}

    for series_number, column in enumerate(
        time_series_columns,
        start=1
    ):
        series = instance_data[column]

        instance_features[f"min{series_number}"] = series.min()
        instance_features[f"max{series_number}"] = series.max()
        instance_features[f"mean{series_number}"] = series.mean()
        instance_features[f"std{series_number}"] = series.std()
        instance_features[f"median{series_number}"] = series.median()
        instance_features[f"1st_quart{series_number}"] = (
            series.quantile(0.25)
        )
        instance_features[f"3rd_quart{series_number}"] = (
            series.quantile(0.75)
        )
    feature_record.append(instance_features)

    instance_record.append({
        "Instance": instance_number + 1,
        "Activity": row["Activity"],
        "Dataset": row["Dataset"],
        "Split": row["Split"],
        "Observations": len(instance_data)
    })

feature_dataset = pd.DataFrame(
    feature_record,
    index=range(1, len(feature_record) + 1 )
)
feature_dataset.index.name = "Instance"

instance_information = pd.DataFrame(
    instance_record,
).set_index("Instance")

print("Feature dataset shape:", feature_dataset.shape)
print("Missing feature values:", feature_dataset.isna().sum().sum())

feature_dataset

Feature dataset shape: (88, 42)
Missing feature values: 0


,min1,max1,mean1,std1,median1,1st_quart1,3rd_quart1,min2,max2,mean2,...,median5,1st_quart5,3rd_quart5,min6,max6,mean6,std6,median6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,1.476967,40.50,39.25,42.0000,0.0,1.30,0.358604,...,35.00,33.0000,36.00,0.00,1.92,0.570583,0.582915,0.43,0.0000,1.3000
2,38.00,45.67,42.812812,1.435550,42.50,42.00,43.6700,0.0,1.22,0.372437,...,33.00,32.0000,34.50,0.00,3.11,0.571083,0.601010,0.43,0.0000,1.3000
3,35.00,47.40,43.954500,1.558835,44.33,43.00,45.0000,0.0,1.70,0.426250,...,36.00,35.3625,36.50,0.00,1.79,0.493292,0.513506,0.43,0.0000,0.9400
4,33.00,47.75,42.179812,3.670666,43.50,39.15,45.0000,0.0,3.00,0.696042,...,35.00,30.4575,36.33,0.00,2.18,0.613521,0.524317,0.50,0.0000,1.0000
5,33.00,45.75,41.678063,2.243490,41.75,41.33,42.7500,0.0,2.83,0.535979,...,30.00,28.4575,31.25,0.00,1.79,0.383292,0.389164,0.43,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,4.650935,34.25,30.25,37.0000,0.0,14.67,4.576562,...,16.00,13.7300,18.25,0.00,8.32,3.259729,1.640243,3.11,2.0500,4.3225
85,19.75,45.50,34.322750,4.752477,35.25,31.00,38.0000,0.0,13.47,4.456333,...,15.75,13.5000,17.75,0.00,9.67,3.432563,1.732727,3.20,2.1575,4.5650
86,19.50,46.00,34.546229,4.842294,35.25,31.25,37.8125,0.0,12.47,4.371958,...,15.75,14.0000,17.75,0.00,10.00,3.338125,1.656742,3.08,2.1600,4.3350


#### 1(c)(iii) - Bootstrap Confidence Intervals

The sample standard deviation of each feature was calculated across the 88 instances. A percentile bootstrap with 10,000 resamples was used to construct a 90% confidence interval for each standard deviation.

In [12]:
number_of_bootstraps = 10000
random_generator = np.random.default_rng(42)

feature_values = feature_dataset.to_numpy(dtype=float)
number_of_instances, number_of_features = feature_values.shape

estimated_standard_deviations = feature_values.std(
    axis=0,
    ddof=1,
)

bootstrap_standard_deviations = np.empty((
    number_of_bootstraps,
    number_of_features,
))

for bootstrap_index in range(number_of_bootstraps):
    sampled_indices = random_generator.choice(
        number_of_instances,
        size=number_of_instances,
        replace=True,
    )

    bootstrap_sample = feature_values[sampled_indices]

    bootstrap_standard_deviations[bootstrap_index] = (
        bootstrap_sample.std(axis=0, ddof=1)
    )

lower_bounds = np.percentile(
    bootstrap_standard_deviations,
    5,
    axis=0,
)

upper_bounds = np.percentile(
    bootstrap_standard_deviations,
    95,
    axis=0,
)

standard_deviation_results = pd.DataFrame({
    "Feature": feature_dataset.columns,
    "Estimated Standard Deviation": estimated_standard_deviations,
    "90% CI Lower Bound": lower_bounds,
    "90% CI Upper Bound": upper_bounds,
}).set_index("Feature")

standard_deviation_results.round(4)

,Estimated Standard Deviation,90% CI Lower Bound,90% CI Upper Bound
Feature,,,
min1,9.5700,8.2360,10.7739
max1,4.3944,3.3399,5.2931
mean1,5.3357,4.7038,5.8758
std1,1.7722,1.5694,1.9495
median1,5.4401,4.7840,6.0063
1st_quart1,6.1536,5.5738,6.6486
3rd_quart1,5.1389,4.3386,5.8340
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6306,5.4011


**Results:** Each bootstrap sample was formed by sampling 88 complete instances with replacement. The standard deviation of every feature was recalculated for each of the 10,000 bootstrap samples.

The 5th and 95th percentiles of the bootstrap estimates form the lower and upper bounds of each 90% confidence interval. Wider intervals indicate greater uncertainty in the estimated standard deviation.

#### 1(c)(iv) - Selection of Important Features

I selected the minimum, maximum, and mean as the three most important time-domain features.

The mean represents the overall level of a sensor signal, while the minimum and maximum describe its lower and upper extremes. Together, these features capture both the typical sensor value and its range of movement. These characteristics may help distinguish stationary activities, such as sitting and lying, from more dynamic activities, such as walking and cycling.

## 2 - ISLR 3.7.4

**/4. I collect a set of data (n = 100 observations) containing a single predictor and a quantitative response. I then fit a linear regression model to the data, as well as a separate cubic regression, i.e. Y = β0 + β1X + β2X 2 + β3X 3 + ".**

**(a) Suppose that the true relationship between X and Y is linear,i.e. Y = β0 + β1X + ". Consider the training residual sum of squares (RSS) for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.**

The cubic regression will have training RSS less than or equal to the linear regression’s training RSS.

The linear model is nested within the cubic model: setting (\beta_2=\beta_3=0) makes the cubic model linear. Therefore, the cubic model can always reproduce the linear fit and may use its additional terms to fit the training data more closely. Its training RSS can never be higher, although it could be equal in a special case.

**(b) Answer (a) using test rather than training RSS.**

We expect the linear regression to have lower test RSS on average.

Because the true relationship is linear, the (X^2) and (X^3) terms do not capture genuine structure. Instead, they may fit random noise in the training data. This gives the cubic model higher variance and can reduce its performance on unseen observations. Therefore, the correctly specified linear model should generally have lower test RSS, although the cubic model could perform better on a particular test sample by chance.

**(c) Suppose that the true relationship between X and Y is not linear, but we don’t know how far it is from linear. Consider the training RSS for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.**

The cubic regression will again have training RSS less than or equal to the linear regression’s training RSS.

Regardless of the true relationship, the cubic model is more flexible and includes the linear model as a special case. It can either use the additional polynomial terms to improve the training fit or set their coefficients to zero. Therefore, its training RSS cannot be greater than that of the linear model.

**(d) Answer (c) using test rather than training RSS.**

There is not enough information to determine which model will have lower test RSS.

The cubic model may have lower bias because it can represent some nonlinear relationships, but it also has higher variance because it is more flexible. If the true relationship is strongly nonlinear and can be approximated well by a cubic function, the cubic model will likely have lower test RSS. If the relationship is close to linear, the linear model may perform better because of its lower variance. The result depends on the form and degree of nonlinearity, the noise level, and the training sample. Cross-validation could be used to compare the models.

